# FlowTrack Colab Notebook

This notebook covers:
1. Environment setup in Google Colab
2. YOLO training (smoke + full-ready config)
3. Model evaluation
4. Inference on EarthCam live HLS stream
5. Visualization with Matplotlib


In [ ]:
# Clone repo (skip if already mounted in your Colab session)
import os
if not os.path.isdir('/content/FlowTrack'):
    !git clone https://github.com/oaboelazm/FlowTrack.git /content/FlowTrack
%cd /content/FlowTrack


In [ ]:
# Install dependencies
!pip -q install --upgrade pip
!pip -q install -r requirements.txt


In [ ]:
import os
import cv2
import math
import torch
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO

print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('Device:', 'cuda:0' if torch.cuda.is_available() else 'cpu')


## Training Config
Use smoke mode first to validate the pipeline quickly, then switch to full training.


In [ ]:
from pathlib import Path
import yaml

device = '0' if torch.cuda.is_available() else 'cpu'

smoke_cfg = {
    'project': 'runs/flowtrack_colab',
    'name': 'visdrone_smoke',
    'model': 'yolov8n.pt',
    'data': 'VisDrone.yaml',
    'epochs': 3,
    'imgsz': 640,
    'batch': 16 if device == '0' else 8,
    'device': device,
    'workers': 2,
    'fraction': 0.05,
    'amp': True if device == '0' else False,
    'pretrained': True,
    'save': True,
    'plots': True
}

full_cfg = {
    'project': 'runs/flowtrack_colab',
    'name': 'visdrone_full',
    'model': 'yolov8s.pt',
    'data': 'VisDrone.yaml',
    'epochs': 40,
    'imgsz': 960,
    'batch': 16 if device == '0' else 8,
    'device': device,
    'workers': 2,
    'amp': True if device == '0' else False,
    'pretrained': True,
    'save': True,
    'plots': True
}

Path('configs/training').mkdir(parents=True, exist_ok=True)
with open('configs/training/colab_smoke.yaml', 'w') as f:
    yaml.safe_dump(smoke_cfg, f, sort_keys=False)
with open('configs/training/colab_full.yaml', 'w') as f:
    yaml.safe_dump(full_cfg, f, sort_keys=False)

print('Wrote configs:')
print('- configs/training/colab_smoke.yaml')
print('- configs/training/colab_full.yaml')


In [ ]:
# Train smoke profile (recommended first run)
import yaml
cfg = yaml.safe_load(open('configs/training/colab_smoke.yaml'))
model_name = cfg.pop('model')
model = YOLO(model_name)
model.train(**cfg)


In [ ]:
# Locate best checkpoint and run validation
from pathlib import Path
import glob

candidates = sorted(glob.glob('runs/flowtrack_colab/**/weights/best.pt', recursive=True))
assert candidates, 'No best.pt found. Check training logs.'
best_pt = candidates[-1]
print('Using:', best_pt)

best_model = YOLO(best_pt)
metrics = best_model.val(data='VisDrone.yaml', imgsz=640, device=('0' if torch.cuda.is_available() else 'cpu'))
print(metrics.results_dict)


In [ ]:
# Optional export
onnx_path = best_model.export(format='onnx', imgsz=640)
print('ONNX:', onnx_path)


## Stream Inference (EarthCam)
Important: EarthCam links are tokenized and expire quickly.
If this URL fails, refresh from EarthCam website and paste a fresh link.


In [ ]:
STREAM_URL = 'https://videos-3.earthcam.com/fecnetwork/hdtimes10.flv/chunklist_w981375959.m3u8?t=vBci5OreTDT5OVZWlrH3hFWPpk6y83Y18ohQ4H190JOFyBpZe72WxgRezoylFmsXZRjHS9kUGRoqEJoNOomfxA%3D%3D&td=202602201131'

# EarthCam typically needs headers to avoid 403
os.environ['OPENCV_FFMPEG_CAPTURE_OPTIONS'] = 'user_agent;Mozilla/5.0|referer;https://www.earthcam.com/'

cap = cv2.VideoCapture(STREAM_URL, cv2.CAP_FFMPEG)
print('Stream opened:', cap.isOpened())


In [ ]:
# Sample N frames from stream and run inference
N = 6
annotated = []

for i in range(N):
    ok, frame = cap.read()
    if not ok or frame is None:
        print(f'Failed frame {i}')
        continue

    results = best_model.predict(source=frame, conf=0.35, iou=0.45, imgsz=640, verbose=False)
    vis = results[0].plot()  # BGR
    vis_rgb = cv2.cvtColor(vis, cv2.COLOR_BGR2RGB)
    annotated.append(vis_rgb)

cap.release()
print('Collected frames:', len(annotated))


In [ ]:
# Show results with Matplotlib
if len(annotated) == 0:
    print('No frames to display. Try a fresh stream token.')
else:
    cols = 3
    rows = math.ceil(len(annotated) / cols)
    plt.figure(figsize=(16, 4 * rows))
    for i, img in enumerate(annotated):
        plt.subplot(rows, cols, i + 1)
        plt.imshow(img)
        plt.axis('off')
        plt.title(f'Frame {i+1}')
    plt.tight_layout()
    plt.show()


## Notes for Better Accuracy
- Switch to `configs/training/colab_full.yaml` for stronger model quality.
- Add BDD100K/custom camera fine-tuning for deployment realism.
- Calibrate traffic line and speed scaling per camera view.
